In [1]:
import pandas as pd
import numpy as np
import pyodbc as odbc

Reading CSV

In [2]:
df = pd.read_csv("../01-data/karachi_ecommerce_orders_dirty.csv")
print(f"Total Rows: ", len(df))

Total Rows:  378


# TRANSFORMATION PHASE

dropping duplicates

In [3]:
initial_rows = len(df)
df = df.drop_duplicates()
final_rows = len(df)
print(f"Total duplicate rows removed: ", initial_rows - final_rows)

Total duplicate rows removed:  5


Data Types & Text Standardization

In [4]:
text_columns = ['city', 'product_category', 'product_name', 'payment_method', 'order_status', 'sales_channel']

for column in text_columns:
    df[column] = df[column].astype(str).str.strip().str.title()

order_date Column Dates Format Standardization

In [5]:
#og dates ko preserve kr lia pehle
original_dates = df['order_date'].copy()

#unko date mn parse krdia
df['order_date'] = pd.to_datetime(
    original_dates,
    format='%m/%d/%Y',
    errors='coerce'
)
#jo dates fail hogyin parse hone se
mask = df['order_date'].isna()

df.loc[mask, 'order_date'] = pd.to_datetime(
    original_dates[mask],
    format='%d/%m/%Y',
    errors='coerce'
)
##dubara se wo dates identify kri jo parse na hoskin eg 2026-08-30
dash_mask = df['order_date'].isna()

df.loc[dash_mask, 'order_date'] = pd.to_datetime(
    original_dates[dash_mask],
    format='%Y-%m-%d',
    errors='coerce'
)
#dubara se wo dates identify kri jo parse na hoskin like 20260920
mask_no_slash = df['order_date'].isna()

df.loc[mask_no_slash, 'order_date'] = pd.to_datetime(
    original_dates[mask_no_slash],
    format='%Y%m%d',
    errors='coerce'
)
#wo rows lelin jahan date null hai. (unko rejected_records_logs.csv file mn save krunga, aur end mn sab variable ko ek single all_error_rows mn pd.concat krdunga then file mn write)
null_date_rows = df[df['order_date'].isna()].copy()
null_date_rows['error_reason'] = 'Missing Order Date'

df = df[df['order_date'].notna()]

#ab string format mn parse kr dia dates ko
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')

## kaam ki chiz ⭐⭐⭐
1. Pehle dhoondhein jahan date null hai aur wajah likhein
null_date_rows = df[df['order_date'].isna()].copy()
null_date_rows['Error_Reason'] = 'Missing Order Date'

2. Phir dhoondhein jahan Customer ID null hai aur wajah likhein
null_cust_rows = df[df['customer_id'].isna()].copy()
null_cust_rows['Error_Reason'] = 'Missing Customer ID'

3. Phir dhoondhein jahan Amount null hai aur wajah likhein
null_amount_rows = df[df['amount'].isna()].copy()
null_amount_rows['Error_Reason'] = 'Missing Amount'

4. PRO-TIP: Saare kharab variables ko EK SINGLE DataFrame mein jor (combine) dein
all_errors_df = pd.concat([null_date_rows, null_cust_rows, null_amount_rows])

5. Pipeline ke end par sirf is EK file ko write karein!
all_errors_df.to_csv('master_etl_error_log.csv', index=False)


Quantity Column Fix

In [6]:
#pehle quantity column ko numeric mn convert kr lia.
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

#then negative quantities ko positive bana dia
df['quantity'] = df['quantity'].abs() 

#error file mn 0 quantity wali rows daal dunga
zero_qtty_rows = df[df['quantity'] == 0 ].copy()
zero_qtty_rows['error_reason'] = 'Invalid Quantity (Zero)'

#null quantity rows ko bhi file mn chipka dunga
null_qtty_rows = df[df['quantity'].isna()].copy()
null_qtty_rows['error_reason'] = 'Null Quantity Found'

df = df[(df['quantity'] > 0) & (df['quantity'].notna())]

unit_price_pkr Column Fix

In [7]:
df['unit_price_pkr'] = pd.to_numeric(df['unit_price_pkr'], errors='coerce')

#neg prices ko positive kr dia
df['unit_price_pkr'] = df['unit_price_pkr'].abs()

#zero price aur null price rows ko file mn daalna
zero_unit_price_rows = df[df['unit_price_pkr'] == 0].copy()
zero_unit_price_rows['error_reason'] = 'Zero Unit Price'

null_unit_price_rows = df[df['unit_price_pkr'].isna()].copy()
null_unit_price_rows['error_reason'] = 'Null Unit Price'

#finally zero aur null price wali rows ko df se remove
df = df[(df['unit_price_pkr'] > 0) & (df['unit_price_pkr'].notna())]

discount_pct Column Fix

In [8]:
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')

df['discount_pct'] = df['discount_pct'].abs()

#filtering out rows that are null (btw my file has no nulls in column discount_pct)
null_discount_rows = df[df['discount_pct'].isna()].copy()
null_discount_rows['error_reason'] = 'Null Discount'

# Agar data % mein hai (e.g. 105%) to > 100, aur agar decimals mein hai to > 1.0 check karein
invalid_discount_rows = df[df['discount_pct'] > 1.0].copy()
invalid_discount_rows['error_reason'] = 'Discount Greater Than 100%'

df = df[(df['discount_pct'] >= 0.0) & (df['discount_pct'] <= 1.0) & (df['discount_pct'].notna())]

delivery_days Column Fix

In [9]:
df['delivery_days'] = pd.to_numeric(df['delivery_days'], errors='coerce')

#sirf negative days ko handle krna hai isme
df['delivery_days'] = df['delivery_days'].abs()

#Ab float se integer ('Int64') mein convert karein
df['delivery_days'] = df['delivery_days'].astype('Int64')

order_line_id column duplicates fix

In [10]:
#Duplicated Primary Key rows ko Identify aur Capture karo (for rejection log)
duplicate_pk_rows = df[df.duplicated(subset=['order_line_id'], keep='first')].copy()

#saving into rejected_records_logs.csv
duplicate_pk_rows['error_reason'] = 'Duplicate Order Line ID (PK Conflict)'

#DataFrame se duplicate order_line_ids ko drop karke pehli row retain karo
df = df.drop_duplicates(subset=['order_line_id'], keep='first')

#### customer_id Column Fix
Direct reject karne se pehle dekho ke kya us same order_id ki kisi doosri line item mein customer_id maujood hai?

- Example: Agar order_id 8003 ki line 1 mein customer_id = C1076 hai aur line 2 mein empty hai, toh iska matlab yeh same customer ka order hai! Tum isey same order_id ke group se fill (impute) kar sakte ho.

In [11]:
#har group k missing rows ko ffill aur bfill se fill krdia
df['customer_id'] = df.groupby('order_id')['customer_id'].ffill().bfill()

#wo customers jinki customer_id fill na hoski (rejected_records_logs.csv mn daal dengy)
null_customer_id_rows = df[df['customer_id'].isna()].copy()
null_customer_id_rows['error_reason'] = 'Missing Customer Id'

#sirf wohi rows rkho jahan customer_id null na ho
df = df[df['customer_id'].notna()]

city Column Fix (same as customer_id)

In [12]:
df['city'] = df.groupby('order_id')['city'].ffill().bfill()

null_city_rows = df[df['city'].isna()].copy()
null_city_rows['error_reason'] = 'Missing City'

df = df[df['city'].notna()]

payment_method column fix

In [13]:
df['payment_method'] = df.groupby('order_id')['payment_method'].ffill().bfill()

null_payment_method_rows = df[df['payment_method'].isna()].copy()
null_payment_method_rows['error_reason'] = 'Missing Payment Method'

df = df[df['payment_method'].notna()]

 product_category Column Fix
- Logic: Agar kisi row mein product_name "Yoga Mat" hai par product_category empty hai, toh dataset mein doosri rows se jahan "Yoga Mat" ki category "Sports" mili hai, wahan se fill ho sakta hai!

In [14]:
df['product_category'] = df.groupby('product_name')['product_category'].ffill().bfill()

null_prod_cat_rows = df[df['product_category'].isna()].copy()
null_prod_cat_rows['error_reason'] = 'Missing Product Category'

df = df[df['product_category'].notna()]

product_name Column Fix (isko recover nhi kia jaskta)

In [15]:
null_prod_name_rows = df[df['product_name'].isna()].copy()
null_prod_name_rows['error_reason'] = 'Missing Product Name'

df = df[df['product_name'].notna()]
print("Congratulations! Your Data is Clean.\nNow, you can insert it into SQL Server.")

Congratulations! Your Data is Clean.
Now, you can insert it into SQL Server.


## Saving cleaned data and Rejected data into a separate file

In [ ]:
#Clean data ko csv mn save dia
df.to_csv("../01-data/02-processed/01-raw/karachi_ecommerce_orders_cleaned.csv" , index=False )
print(f"Successfully inserted data into karachi_ecommerce_orders_cleaned.csv")

rows = df.shape[0]
columns = df.shape[1]
print(f"No of rows inserted: {rows}")
print(f"No of columns inserted: {columns}")

Successfully inserted data into karachi_ecommerce_orders_cleaned.csv
No of rows inserted: 360
No of columns inserted: 14


In [17]:
#Pehle sab variables ko concat krdia
rejected_data = pd.concat(
    [null_date_rows, 
    zero_qtty_rows, 
    null_qtty_rows, 
    zero_unit_price_rows,
    null_unit_price_rows,
    null_discount_rows,
    invalid_discount_rows,
    null_customer_id_rows, 
    null_city_rows,
    null_payment_method_rows,
    duplicate_pk_rows])
#Rejected data ko rejected_records_logs.csv mn daal dia
rejected_data.to_csv("../01-data/03-rejected/rejected_records_logs.csv")

print(f"Successfully inserted data into rejected_records_logs.csv")

rows = rejected_data.shape[0]
columns = rejected_data.shape[1]

print(f"No of rows inserted: {rows}")
print(f"No of columns inserted: {columns}")

Successfully inserted data into rejected_records_logs.csv
No of rows inserted: 12
No of columns inserted: 15


### use before inserting
- Insert karne se pehle NaN values ko Python None mein convert kar lo
- df_sql = df.where(pd.notnull(df), None)

- Ab tuples ki list banao
- records_to_insert = [tuple(x) for x in df_sql.to_numpy()]

## Making Data Ready for Insertion

In [18]:
#Pehle Object type mein cast karo, phir saare NaNs / pd.NA ko None (SQL NULL) banao
df_sql = df.astype(object).where(pd.notnull(df), None)

data_to_insert = []
for row in df_sql.to_numpy():
    data_to_insert.append(tuple(row))

## SQL Insertion

In [19]:
SERVER_NAME = 'HAIDERs-LAPTOP'
DATABASE_NAME = 'Karachi_ecommerce'

connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_connection=yes;"
)
try:
    conn = odbc.connect(connection_string)
    cursor = conn.cursor()
    #parameterized query
    insert_query = """
        INSERT INTO orders(order_id,order_line_id,order_date,customer_id,city,product_category,product_name,quantity,unit_price_pkr,discount_pct,payment_method,order_status,delivery_days,sales_channel)
        VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """
    #cursor.execute("TRUNCATE TABLE orders")
    cursor.executemany(insert_query, data_to_insert)

    conn.commit()

    cursor.close()
    conn.close()
    print(f"Your Data has been successfully inserted into Sql")
except Exception as e:
    print("Oops! An error Occured: ", e)

Your Data has been successfully inserted into Sql


## README.md

Yeh ek bohot hi zabardast aur structured repository layout hai ek beginner project ke hissaab se! Folder numbering (01-data, 02-scripts) isko bohot clean aur professionally readable banati hai.
Aapke points ke mutabiq, aapki specific requirements ko step-by-step clear aur details ke sath cover karte hain:

1. Architecture Diagram Hint (Illustrator ke liye)

Kyunki aap Illustrator use kar rahe hain, aap ek clean Left-to-Right linear flow design karein. Graphic elements ke liye aap standard tools ke icons use kar sakte hain:
- Source Blocks: Ek "File" ka icon banayein jisme karachi_ecommerce_orders_dirty.csv likha ho.
- Processing Blocks: Ek central box banayein jise "Pandas/Numpy ETL Engine" ka naam dein. Iske do sub-arrows nikalen:
	- Ek arrow neeche ki taraf jaye jo "Rejected Logs File" (rejected_records_logs.csv) ko point kare.
	- Ek main arrow aage jaye jo "Cleaned Data" (karachi_ecommerce_orders_clean.csv) ko point kare.
- Storage/Database Block: Ek standard Cylindrical Database icon banayein jo aapke SQL Database ko represent kare (jahan data load ho raha hai). Iske upar 01-create-table.sql ka flowchart point kar raha ho.
- Analytics/Reporting Block: SQL Database se ek arrow nikal kar teen alag-alag image frames/dashboards par jaye (Aapki teeno .png files ke liye: category_revenue_summary, daily_revenue_trend, order_status_distribution).

3. Data Pipeline Architecture (The ETL Flow)

Aapki README ke is section mein aapko pipeline ka technical flow step-by-step breakdown karna hoga. Aap is tarah se likh sakte hain:
• Extraction (E):
	• In-gestion of raw Karachi E-commerce transaction data (karachi_ecommerce_orders_dirty.csv) using Pandas.
• Transformation & Data Quality (T):
	• Data Cleaning: Handling missing values, parsing dates, correcting data types, and stripping whitespaces using Pandas and NumPy.
	• Data Quality Filter / Isolation Layer: Rows breaking critical business rules (e.g., negative order amounts, missing customer IDs) are systematically isolated and routed to rejected_records_logs.csv without breaking the main execution loop.
	• Transformation: Aggregating specific dimensions for performance-ready loading.
• Loading (L):
	• Establishing connection to the SQL Database via Python.
	• Executing 01-create-table.sql to enforce strictly typed destination schemas.
	• Writing clean records into the final operational tables.
• Analytics & Reporting:
	• Executing advanced analytical queries (02-revenue_and_order_metrics.sql) on the structured database.
	• Using Matplotlib to automatically export daily revenue trends, category health charts, and order distributions as operational PNG reports.


4. Repository Structure

Since you have already built a fantastic structure (as seen in your screenshot), you should format it cleanly using a standard Markdown text block in your README. Here is the exact copy-pasteable structure of your project:
text
├── 01-data/
│   ├── 01-raw/
│   │   └── karachi_ecommerce_orders_dirty.csv      # Raw incoming source data
│   ├── 02-processed/
│   │   └── karachi_ecommerce_orders_clean.csv    # Sanitized data ready for SQL loading
│   └── 03-rejected/
│       └── rejected_records_logs.csv             # Segregated bad rows for debugging
├── 02-scripts/
│   ├── transform_clean.py                        # Pandas/NumPy cleaning logic
│   ├── load_sql.py                               # Database ingestion layer
│   ├── generate_reports.py                       # Matplotlib reporting automation
│   └── etl_pipeline.py                           # Master controller script
├── 03-sql/
│   ├── 01-create-table.sql                       # Target schema definition
│   └── 02-revenue_and_order_metrics.sql          # Analytical queries for reports
├── 04-notebooks/
│   ├── etl_pipeline.ipynb                        # Step-by-step pipeline prototyping
│   └── reports_prototyping.ipynb                 # Visualization experimentation
├── 05-images/
│   ├── category_revenue_summary.png              # Output business report 1
│   ├── daily_revenue_trend.png                   # Output business report 2
│   └── order_status_distribution.png             # Output business report 3
└── README.md
Use code with caution.

5. Prerequisites & Setup Instructions

Even though you are writing this yourself, here is a quick layout framework you can follow to keep it punchy and clear for anyone cloning your repository:
• Prerequisites:
	• Python 3.8+ installed.
	• Local SQL database instance active (e.g., PostgreSQL / MySQL / SQLite).
• Quick Start Steps:
	1. Clone Repo: git clone <your-repo-url>
	2. Install Dependencies: pip install -r requirements.txt (Make sure to generate this file using pip freeze > requirements.txt).
	3. Setup Database: Run the scripts inside 03-sql/01-create-table.sql on your local SQL environment.
	4. Execute Pipeline: Run python 02-scripts/etl_pipeline.py to trigger the entire flow from extraction to visual reporting.

6. Multi-Cloud Challenges & Solutions (Kyunki aap Multi-Cloud Role aim kar rahe hain)

Kyunki aap ek Multi-Cloud Data Engineer ke taur par grow karna chahte hain, aap is local project ko ek complete enterprise multi-cloud blueprint ki tarah README mein pitch kar sakte hain. Aap README mein "How this Scales to Multi-Cloud" ya "Future Multi-Cloud Architecture Strategy" ka section daalein:
• Challenge 1: Cloud Egress Costs and Multi-Cloud Ingestion
	• Scenario: Agar Data AWS S3 par land ho raha ho aur central warehouse Google Cloud (BigQuery) ya Azure Synapse par ho, to clouds ke darmiyan data fetch karne par connectivity cost aur latency lagti hai.
	• Solution: We can implement localized staging. Hum script ko AWS Lambda par run karke intermediate outputs ko vahin compress (Parquet format mein) karenge, taaki cross-cloud network transfer minimal ho.
• Challenge 2: Cross-Cloud Security and Identity Federation
	• Scenario: Local python scripts mein load_sql.py local credentials directly read kar leti hai, par multi-cloud infrastructure mein systems ko aapas mein secure data pipeline chalane ke liye direct passwords nahi diye ja sakte.
	• Solution: Transitioning from local auth to Workload Identity Federation or Cloud Secret Managers (like AWS Secrets Manager to safely passing temporary keys to Azure/GCP systems) ensuring no hardcoded strings or security leakage.
• Challenge 3: Multi-Cloud Orchestration Split
	• Scenario: Alag cloud triggers ko handle karna jab files alag environments mein aati hain.
	• Solution: Utilizing a cloud-agnostic platform like Apache Airflow or Prefect to natively bridge and monitor tasks executing across AWS nodes and GCP data structures simultaneously.

7. Monitoring, Logging & Future Enhancements

• Logging Strategy: Aap likhein ke aapne native Python logging module deploy kiya hai jo script execution ke process ko log karta hai (Info, Warning, Error stages) taaki database constraints fail hone par foran traceability miley.
• Future Enhancements (Aapki Next Steps List):
	• Containerization: Dockerize the entire python ETL execution block so it runs anywhere identically.
	• Infrastructure as Code (IaC): Add Terraform code to provision the SQL database and storage paths automatically.
	• CI/CD Pipeline: Implement GitHub Actions to run automated testing (pytest) on code push.
Neeche is pooray structure ka ek unified comparison table diya gaya hai taaki aapko pata ho ke local architecture multi-cloud enterprise mein kaise transform hoti hai:

Attribute / Step	Current Local Implementation	Enterprise Multi-Cloud Scaling Blueprint
Data Landing (Extract)	Local Directory (01-raw)	AWS S3 Bucket / Azure Blob Storage
Processing (Transform)	Local Python Scripts (pandas/numpy)	PySpark running on AWS EMR or Databricks
Data Quality Filter	Row Isolation to rejected_records_logs.csv	Automated Dead-Letter Queue (DLQ) with alert notifications
Data Loading (Load)	Local SQL Scripts (.sql)	Snowflake Data Warehouse / Google BigQuery
Visualization (BI)	Static Matplotlib Charts (.png)	Dynamic Dashboards via PowerBI / Looker Studio
Security & Secrets	Local environment variables	AWS Secrets Manager / GCP Secret Manager / Workload Identity

Agar aap chahein, to main is pure data flow ko exact Markdown Code Block mein likh kar de sakta hoon taaki aap use direct copy karke apni README.md file mein paste kar sakein. Kya main aapke liye woh file draft karoon?


## READY MADE README.md (but not recommended)

# Karachi E-Commerce End-to-End ETL & Reporting Pipeline

An automated data engineering pipeline that extracts raw, volatile e-commerce transactional data, applies production-grade data quality transformations, loads the sanitized records into a relational database, and automatically generates business intelligence visual reports.

---

## 1. Project Architecture Diagram
*Note: Place your Adobe Illustrator exported architecture diagram image in `05-images/architecture.png` and reference it here.*

---

## 2. Project Overview & Business Problem
Real-world e-commerce transactional systems ingest millions of records containing missing entries, formatting anomalies, and duplicate entries. This project addresses these challenges by building a robust local data engine that isolates anomalies, schema-enforces incoming records, and delivers accurate revenue analytics and operational insights for the Karachi retail region.

### Tech Stack
*   **Language:** Python 3.8+
*   **Data Processing:** Pandas, NumPy
*   **Storage & Database:** Relational SQL Database (PostgreSQL/MySQL/SQLite)
*   **Data Visualization:** Matplotlib
*   **Prototyping:** Jupyter Notebooks

---

## 3. Data Pipeline Architecture (The ETL Flow)

*   **Extraction (E):** Ingests raw transactional data (`karachi_ecommerce_orders_dirty.csv`) utilizing Pandas chunk-based file handling strategies.
*   **Transformation & Data Quality Filter (T):**
    *   **Data Cleaning:** Normalizes string structures, standardizes date formats, handles null records, and performs strict datatype conversions.
    *   **Isolation Layer (Data Dead-Letter Queue):** Rows violating foundational business metrics (e.g., negative revenue values or missing key identifiers) are securely routed out of the primary pipe into `rejected_records_logs.csv` for post-mortem debugging without halting execution.
*   **Loading (L):** Establishes an ingestion connection to the relational database to apply strict target schema constraints via `01-create-table.sql` before streaming optimized rows.
*   **Analytics & Visual Reporting:** Triggers structured financial queries (`02-revenue_and_order_metrics.sql`) to pull metrics, auto-generating and saving production-ready graphical reports into the `05-images/` directory.

---

## 4. Repository Structure

```text
├── 01-data/
│   ├── 01-raw/
│   │   └── karachi_ecommerce_orders_dirty.csv      # Raw incoming source data
│   ├── 02-processed/
│   │   └── karachi_ecommerce_orders_clean.csv    # Sanitized data ready for SQL loading
│   └── 03-rejected/
│       └── rejected_records_logs.csv             # Segregated bad rows for debugging
├── 02-scripts/
│   ├── transform_clean.py                        # Pandas/NumPy cleaning logic
│   ├── load_sql.py                               # Database ingestion layer
│   ├── generate_reports.py                       # Matplotlib reporting automation
│   └── etl_pipeline.py                           # Master controller script
├── 03-sql/
│   ├── 01-create-table.sql                       # Target schema definition
│   └── 02-revenue_and_order_metrics.sql          # Analytical queries for reports
├── 04-notebooks/
│   ├── etl_pipeline.ipynb                        # Step-by-step pipeline prototyping
│   └── reports_prototyping.ipynb                 # Visualization experimentation
├── 05-images/
│   ├── category_revenue_summary.png              # Output business report 1
│   ├── daily_revenue_trend.png                   # Output business report 2
│   └── order_status_distribution.png             # Output business report 3
└── README.md
```

---

## 5. Prerequisites & Setup Instructions

### Prerequisites
*   Python 3.8 or higher installed on your system.
*   An active local instance of a SQL database (e.g., PostgreSQL / MySQL).

### Quick Start Guide

1. **Clone the Repository:**
   ```bash
   git clone <your-repository-url>
   cd ETL-Project
   ```

2. **Install Required Packages:**
   ```bash
   pip install -r requirements.txt
   ```

3. **Initialize Database Schema:**
   Execute the contents of `03-sql/01-create-table.sql` in your local database management workspace to build the destination landscape.

4. **Run the ETL Pipeline:**
   Execute the master orchestration controller script to trigger the full processing chain:
   ```bash
   python 02-scripts/etl_pipeline.py
   ```

---

## 6. Enterprise Multi-Cloud Scaling Blueprint
As a Multi-Cloud Data Engineer, this pipeline has been architected locally with components mapping directly to enterprise, vendor-agnostic cloud architectures:

*   **Cloud Ingestion & Storage:** Scaling out the local `01-data` directory into distributed cloud stores like **AWS S3** or **Azure Blob Storage**, triggering event-driven pipelines on new file landings.
*   **Distributed Processing Layer:** Swapping out centralized memory Pandas processing with scalable cluster engines like **PySpark** running on **AWS EMR** or **Azure Databricks** to process massive data footprints.
*   **Enterprise Warehousing:** Moving from a localized relational setup to cloud-agnostic data platforms like **Snowflake** or native structures like **Google BigQuery** to handle modern warehouse computing scales.
*   **Security & Governance Frameworks:** Transitioning local credentials to automated federated cloud environments using **AWS Secrets Manager** or **GCP Secret Manager** with cross-cloud identity models.

---

## 7. Monitoring, Logging & Future Enhancements

*   **Robust Logging Framework:** Employs the native Python `logging` module to keep granular system runtime records. Every execution stage (Success, Warning, Exception tracking) maps errors down to explicit database schema mismatch traces.
*   **Future Architectural Enhancements:**
    *   **Containerization:** Wrapping the workspace runtime environment inside a **Docker** container to guarantee identical multi-system compilation behavior.
    *   **Infrastructure as Code (IaC):** Standardizing resource generation scripts by creating **Terraform** manifests to configure multi-cloud target buckets and DB resources automatically.
    *   **Orchestration Upgrades:** Migrating execution loops into unified cross-platform workflow tools like **Apache Airflow** or **Prefect** to track dynamic dependencies.

----
****
----

## FILE PATH ERROR SOLUTION
import os
import pandas as pd

# 1. Yeh line aapki is file ki maujuda location (absolute path) nikalegi
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) 

# 2. Ab is BASE_DIR ko use karte hue apna direct path join karlein
RAW_DATA_PATH = os.path.join(BASE_DIR, '01-data', '01-raw', 'karachi_ecommerce_orders_dirty.csv')
CLEAN_DATA_PATH = os.path.join(BASE_DIR, '01-data', '02-processed', 'karachi_ecommerce_orders_clean.csv')

# 3. Ab aap directly is variable ko pandas mein pass kar sakte hain
df = pd.read_csv(RAW_DATA_PATH)
